# 02 · Train and evaluate one or more runs
**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**. About 25-35 min per run.

| Who | Runs |
|---|---|
| First, once | `pilot` (about 15 min) - send the printed inflation check before the real runs |
| Dawar | `seed1`, `seed2`, `seed3` |
| Armughan | `control`, `qonly`, `dilution`, `rank8` |
| Optional, if time | `fullft160m`, `rank64`, `filler10m` |

Each run: builds the training mix (FineWeb-Edu + contaminated copies), fine-tunes Pythia-410M with LoRA, scores every item,
runs the 5 detectors (Perplexity, Zlib, Min-K%, Min-K%++, FSD) and the 13-gram reference check.

**Output:** `results_<run>.zip` per run. Download, unzip into the repo folder, double-click `commit_and_push.bat`.
Notebook 01's files must already be on GitHub.

In [ ]:
# ===== CHOOSE THE RUNS FOR THIS SESSION =====
RUNS_TO_DO = ["pilot"]          # e.g. ["seed1", "seed2", "seed3"]  or  ["control", "qonly", "dilution", "rank8"]

In [ ]:
# --- Get the project code from GitHub (works for public repos; for a private repo add a Kaggle
#     secret called GITHUB_TOKEN: Add-ons > Secrets) ---
import os, subprocess, sys
from pathlib import Path
REPO_URL = "https://github.com/DawarAhmed27/DAV-Reseach-DataContamination.git"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = WORK / "repo"
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if REPO.exists():   # update to the latest GitHub version (local generated files are replaced by GitHub's copy)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO), "reset", "--quiet", "--hard", "@{u}"], check=False)
else:
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", url, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
print("Code ready in", REPO)

In [ ]:
# --- Libraries that Kaggle does not always have ---
!pip install -q peft sentence-transformers 2>/dev/null | tail -1
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU T4 in Settings > Accelerator")

In [ ]:
assert (REPO / "data" / "items.csv").exists(), "data/items.csv not found - run notebook 01 and push its outputs first"
from dcl import pipeline as P
import zipfile, traceback
for run in RUNS_TO_DO:
    print("=" * 70)
    try:
        P.run_experiment(run, root=REPO)
        z = WORK / f"results_{run}.zip"
        with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:
            for p in (REPO / "results").glob(f"{run}*"):
                zf.write(p, p.relative_to(REPO))
        print("Saved", z)
    except Exception:
        traceback.print_exc()
        print(f"Run {run} failed - copy the error above and send it.")